# Phase 2: Data Cleaning & Preprocessing

Data cleaning is a critical step in the data analysis pipeline. In this phase, we transform our 'dirty' raw data into a reliable format for analysis. We will handle missing values, duplicates, outliers, and standardize formats to ensure the integrity of our insights.

In [ ]:
import pandas as pd
import numpy as np
import os

# Setting display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## 1. Loading the Dataset
We load the dataset identified for cleaning. Note: Ensure the file name matches your raw data source.

In [ ]:
# Path to the dirty dataset
file_path = '../data/raw/modified_dirty_dataset.csv'

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
else:
    # Fallback to the available raw data if name differs
    print(f"Warning: {file_path} not found. Please check filename.")
    # df = pd.read_csv('../data/raw/your_actual_file.csv') # Uncomment and adjust if needed

## 2. Initial Dataset Overview
Before cleaning, we check the baseline state of our data.

In [ ]:
print("--- Data Head ---")
display(df.head())

print("\n--- Data Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

## 3. Handling Missing Values

### A. Drop rows where CustomerID is missing
`CustomerID` is our primary identifier for behavioral analysis. Without it, the transaction cannot be attributed to a specific user, making it unusable for CRM insights.

In [ ]:
if 'CustomerID' in df.columns:
    df = df.dropna(subset=['CustomerID'])
    print("Rows with missing CustomerID dropped.")

### B. Fill other missing values
For numerical columns, we use the median to avoid bias from outliers. For categorical columns, we use 'Unknown' or the mode.

In [ ]:
# Fill numerical nulls with median
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical nulls with 'Unknown'
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

print("Remaining missing values:", df.isnull().sum().sum())

## 4. Removing Duplicate Records
Duplicate records inflate sales metrics and misrepresent customer activity.

In [ ]:
initial_count = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_count - len(df)} duplicate rows.")

## 5. Standardizing Text Columns
To ensure consistent grouping (e.g., 'London' vs 'london '), we normalize all text data.

In [ ]:
for col in cat_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()
print("Text columns standardized to lowercase and stripped of extra spaces.")

## 6. Handling Outliers

### A. Logical Filtering
Transactions with `Quantity <= 0` are likely returns or data errors that shouldn't be counted in standard revenue analysis.

In [ ]:
if 'Quantity' in df.columns:
    df = df[df['Quantity'] > 0]
    print("Negative and zero quantities removed.")

### B. Statistical Outlier Removal (IQR)
Extreme values in `Price` or `Quantity` can skew our average calculations.

In [ ]:
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Apply to Price and Quantity if they exist
for col in ['Price', 'Quantity']:
    if col in df.columns:
        df = remove_outliers_iqr(df, col)
        print(f"Outliers removed from '{col}'.")

## 7. Date Format Conversion
Converting `TransactionDate` to datetime enables time-series features like 'Month', 'Day of Week', etc.

In [ ]:
if 'TransactionDate' in df.columns:
    df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
    print("TransactionDate converted to datetime format.")

## 8. Feature Engineering: Revenue
Creating a calculated column for total revenue per line item.

In [ ]:
if 'Quantity' in df.columns and 'Price' in df.columns:
    # Standard calculation: Quantity * Price
    # Note: If your data includes a Discount column, you might subtract it here.
    df['Revenue'] = df['Quantity'] * df['Price']
    print("New 'Revenue' column created.")

## 9. Final Validation
Verifying that our dataset is now clean.

In [ ]:
print("Null values check:", df.isnull().sum().sum())
print("Duplicate rows check:", df.duplicated().sum())
print(f"Final shape of cleaned data: {df.shape}")

## 10. Saving the Cleaned Dataset
Persisting the data for Phase 3 (EDA).

In [ ]:
output_path = '../data/processed/cleaned_data.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")